# 14. Data quality and descriptive trends

## tl;dr

The processing layers pass their structural integrity checks, but buyer identifiers, duration, amount, and independent reference validation remain material limitations. Quarterly PELT results are descriptive break signals, not causal explanations or forecasts.

## Context & Methods

The unit is one awarded Grand Ouest digital procurement episode. The trend window starts at 2015Q2 because the raw extract begins in March 2015. For segment $s$ and quarter $q$ the series is the count $N_{s,q} = \sum_i \mathbf{1}(S_i = s,\, Q_i = q)$, including zero-count quarters.

These are time-series methods and are stated in their natural forms. Only the HMM output is genuinely a conditional probability, and only it is written as one.

**PELT — when did the level shift?** Over the number of change points $K$ and their positions $\tau_1 < \cdots < \tau_K$, PELT minimises

$$\sum_{k=0}^{K} \mathcal{C}\big(y_{\tau_k+1:\tau_{k+1}}\big) + K\beta,\qquad \beta = \lambda\log(n),$$

on the z-standardized series, with $\mathcal{C}$ the within-segment squared error. The first term rewards fitting each segment well; the second charges a fixed price per break, which is what stops the optimum from putting a break between every pair of quarters. Sensitivity multipliers $\lambda \in \{0.5, 1, 2\}$ are run and a break is called **stable** only if it appears within about one quarter under all three. PELT answers *when* the statistical level changed; it never answers *why*.

**ADF and KPSS — is the series stationary?** The two tests are run together because their nulls are opposite:

$$H_0^{\mathrm{ADF}}:\ \text{unit root (non-stationary)}, \qquad H_0^{\mathrm{KPSS}}:\ \text{level-stationary}.$$

Rejecting the ADF null while failing to reject the KPSS null is coherent evidence of stationarity. When they disagree the honest report is **ambiguity** over this short window, not a forced binary label.

**HMM — what regime is the series in now?** A 3-state Gaussian hidden Markov model is fit on the quarter-over-quarter change $\Delta N_t = N_t - N_{t-1}$, not the level. The hidden state $Z_t \in \{\text{decline}, \text{plateau}, \text{growth}\}$ evolves through transition probabilities $P(Z_t = k \mid Z_{t-1} = l)$, and the reported quantity is the posterior

$$P(Z_t = k \mid \Delta N_1, \dots, \Delta N_t),$$

read as: given the observed sequence of quarterly changes **and the fitted model**, how probable is regime $k$ in the current quarter? This is model-conditional. A high posterior on `growth` says the model finds that regime most consistent with recent changes — not that the market is objectively growing.

**OLS — what direction are the last three years?** The recent-direction label comes from

$$N_t = \alpha + \beta t + \varepsilon_t$$

fitted over the latest 12 quarters, where $\hat\beta$ is the estimated change in awarded episodes per quarter. A segment is labelled `increasing` or `decreasing` only when the two-sided **raw** p-value falls below the pre-declared exploratory $\alpha = 0.10$; otherwise it is `stable_or_uncertain`. Because five series are fitted and read together, the signal matrix also carries Holm and Benjamini-Hochberg adjusted p-values across those five tests, and a directional label whose Holm p exceeds $\alpha$ is reported as a nominal signal to monitor rather than as a finding. $\hat\beta$ describes the window it was fitted on. It is **not** a forecast.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data/processed/boamp'
with open(PROCESSED / 'data_quality_profile.json', encoding='utf-8') as f:
    quality = json.load(f)
with open(PROCESSED / 'trend_analysis_summary.json', encoding='utf-8') as f:
    trend_summary = json.load(f)
quarterly = pd.read_csv(PROCESSED / 'trend_quarterly.csv', parse_dates=['quarter_start'])
breakpoints = pd.read_csv(PROCESSED / 'trend_breakpoints.csv')
signals = pd.read_csv(PROCESSED / 'trend_signal_matrix.csv')


## Data

In [ ]:
pd.DataFrame([
    {'metric': 'standardized notices', 'value': quality['volume']['standardized_notices']},
    {'metric': 'reconstructed episodes', 'value': quality['volume']['reconstructed_episodes']},
    {'metric': 'study cohort episodes', 'value': quality['volume']['survival_cohort_rows']},
    {'metric': 'candidate pairs', 'value': quality['volume']['candidate_pairs']},
    {'metric': 'primary successor events', 'value': quality['volume']['accepted_primary_links']},
])

In [ ]:
pd.Series(quality['cohort_missingness'], name='missing_rate').sort_values(ascending=False).to_frame()

## Results

In [ ]:
display(signals)
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, (segment, group) in zip(axes.flatten(), quarterly.groupby('segment', sort=False)):
    group = group.sort_values('quarter_start')
    ax.plot(group['quarter_start'], group['episode_count'], color='#356E9A')
    ax.set_title(segment)
    ax.grid(axis='y', alpha=0.25)
axes.flatten()[-1].axis('off')
fig.suptitle('Quarterly awarded digital procurement episodes')
plt.tight_layout()

In [ ]:
overall = quarterly.loc[quarterly['segment'].eq('Overall')].sort_values('quarter_start')
ax = overall.plot(x='quarter_start', y='duration_completeness', figsize=(10, 4), legend=False, color='#356E9A')
ax.set_title('Reliable duration completeness by quarter')
ax.set_ylabel('share')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.25)

## Takeaways

- The cohort has enough episodes for descriptive survival analysis, but the event definition remains linkage-conditioned.
- Duration missingness changes sharply over time, so global duration imputation would create unsupported temporal structure.
- PELT breaks are candidates for documentary interpretation, not causal findings: the objective above dates a level shift, it does not explain one.
- The HMM regime is a posterior $P(Z_t = k \mid \Delta N_1, \dots, \Delta N_t)$ under the fitted model, not an observed property of the market, and it need not agree with the 12-quarter OLS slope.
- Reference metrics are regional reference-sample evidence; independent specialist review is still needed before any external accuracy claim.